In [1]:
# 기본 라이브러리
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()

# 그래프 기본 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False
import gc


# 인코더 추가
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
#VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor
#상수항추가
from statsmodels.tools.tools import add_constant
# 카이제곱, ANOVA
from scipy.stats import chi2_contingency
from scipy.stats import f_oneway
#Turkeyhsd
from statsmodels.stats.multicomp import pairwise_tukeyhsd

### 데이터 불러오기

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# parquet 파일 데이터를 읽어온다.
# x 값으로 쓸 데이터
df1_train = pd.read_parquet('/content/drive/MyDrive/3.승인매출정보/201807_train_승인매출정보.parquet')
df2_train = pd.read_parquet('/content/drive/MyDrive/3.승인매출정보/201808_train_승인매출정보.parquet')
df3_train = pd.read_parquet('/content/drive/MyDrive/3.승인매출정보/201809_train_승인매출정보.parquet')
df4_train = pd.read_parquet('/content/drive/MyDrive/3.승인매출정보/201810_train_승인매출정보.parquet')
df5_train = pd.read_parquet('/content/drive/MyDrive/3.승인매출정보/201811_train_승인매출정보.parquet')
df6_train = pd.read_parquet('/content/drive/MyDrive/3.승인매출정보/201812_train_승인매출정보.parquet')

In [4]:
# 결과데이터를 불러온다.
segment_df = pd.read_csv('/content/drive/MyDrive/Segment.csv')
segment_df

,기준년월,ID,Segment
0,201807,TRAIN_000000,D
1,201807,TRAIN_000001,E
2,201807,TRAIN_000002,C
3,201807,TRAIN_000003,D
4,201807,TRAIN_000004,E
...,...,...,...
2399995,201812,TRAIN_399995,E
2399996,201812,TRAIN_399996,D
2399997,201812,TRAIN_399997,C
2399998,201812,TRAIN_399998,E


In [5]:
# 데이터프레임의 컬럼 이름을 리스트에 담는다.
column_list = df1_train.columns.tolist()

In [7]:
# 슬라이싱
column_list = column_list[:130]

In [8]:
print(column_list)

['기준년월', 'ID', '최종이용일자_기본', '최종이용일자_신판', '최종이용일자_CA', '최종이용일자_카드론', '최종이용일자_체크', '최종이용일자_일시불', '최종이용일자_할부', '이용건수_신용_B0M', '이용건수_신판_B0M', '이용건수_일시불_B0M', '이용건수_할부_B0M', '이용건수_할부_유이자_B0M', '이용건수_할부_무이자_B0M', '이용건수_부분무이자_B0M', '이용건수_CA_B0M', '이용건수_체크_B0M', '이용건수_카드론_B0M', '이용금액_일시불_B0M', '이용금액_할부_B0M', '이용금액_할부_유이자_B0M', '이용금액_할부_무이자_B0M', '이용금액_부분무이자_B0M', '이용금액_CA_B0M', '이용금액_체크_B0M', '이용금액_카드론_B0M', '이용후경과월_신용', '이용후경과월_신판', '이용후경과월_일시불', '이용후경과월_할부', '이용후경과월_할부_유이자', '이용후경과월_할부_무이자', '이용후경과월_부분무이자', '이용후경과월_CA', '이용후경과월_체크', '이용후경과월_카드론', '이용건수_신용_R12M', '이용건수_신판_R12M', '이용건수_일시불_R12M', '이용건수_할부_R12M', '이용건수_할부_유이자_R12M', '이용건수_할부_무이자_R12M', '이용건수_부분무이자_R12M', '이용건수_CA_R12M', '이용건수_체크_R12M', '이용건수_카드론_R12M', '이용금액_일시불_R12M', '이용금액_할부_R12M', '이용금액_할부_유이자_R12M', '이용금액_할부_무이자_R12M', '이용금액_부분무이자_R12M', '이용금액_CA_R12M', '이용금액_체크_R12M', '이용금액_카드론_R12M', '최대이용금액_일시불_R12M', '최대이용금액_할부_R12M', '최대이용금액_할부_유이자_R12M', '최대이용금액_할부_무이자_R12M', '최대이용금액_부분무이자_R12M', '최대이용금액_CA_R12M', '최대이용금액_체크_R12M', '

In [9]:
df1_train = df1_train[column_list]

In [10]:
df2_train = df2_train[column_list]
df3_train = df3_train[column_list]
df4_train = df4_train[column_list]
df5_train = df5_train[column_list]
df6_train = df6_train[column_list]

In [11]:
combined_df = pd.concat([df1_train, df2_train, df3_train, df4_train, df5_train, df6_train],
                         axis=0,        # 행 방향
                         ignore_index=True)  # 인덱스 초기화

In [12]:
segment_df = segment_df['Segment']

In [13]:
# 세그먼트와 데이터를 합쳐준다.

all_df = pd.concat([segment_df,combined_df], axis = 1)
all_df

,Segment,기준년월,ID,최종이용일자_기본,최종이용일자_신판,최종이용일자_CA,최종이용일자_카드론,최종이용일자_체크,최종이용일자_일시불,최종이용일자_할부,...,이용개월수_신판_R3M,이용개월수_일시불_R3M,이용개월수_할부_R3M,이용개월수_할부_유이자_R3M,이용개월수_할부_무이자_R3M,이용개월수_부분무이자_R3M,이용개월수_CA_R3M,이용개월수_체크_R3M,이용개월수_카드론_R3M,이용가맹점수
0,D,201807,TRAIN_000000,20180719,20180713,20180719,10101,20180203,20180709,20180713,...,3,3,1,0,1,0,3,0,0,6
1,E,201807,TRAIN_000001,20180719,20180719,20170728,20170327,10101,20180719,20171231,...,3,3,0,0,0,0,0,0,0,32
2,C,201807,TRAIN_000002,20180706,20180706,20180706,20151119,20141230,20180706,20180627,...,3,3,1,0,1,0,2,0,0,27
3,D,201807,TRAIN_000003,20180721,20180715,20180721,10101,20141111,20180704,20180715,...,3,3,2,1,1,0,3,0,0,10
4,E,201807,TRAIN_000004,20180124,20180124,10101,10101,20180512,20180124,10101,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399995,E,201812,TRAIN_399995,20181220,20181220,10101,10101,20181212,20181220,20160501,...,0,0,0,0,0,0,0,3,0,3
2399996,D,201812,TRAIN_399996,20181202,20181202,10101,20170112,10101,20181202,20180112,...,3,3,0,0,0,0,0,0,0,15
2399997,C,201812,TRAIN_399997,20181230,20181230,10101,10101,20131124,20181230,20180919,...,3,3,1,0,1,0,0,0,0,27
2399998,E,201812,TRAIN_399998,20161224,20161224,10101,10101,10101,20161224,20150122,...,0,0,0,0,0,0,0,0,0,0


In [14]:
print(all_df.isna().sum().to_string())

Segment               0
기준년월                  0
ID                    0
최종이용일자_기본             0
최종이용일자_신판             0
최종이용일자_CA             0
최종이용일자_카드론            0
최종이용일자_체크             0
최종이용일자_일시불            0
최종이용일자_할부             0
이용건수_신용_B0M           0
이용건수_신판_B0M           0
이용건수_일시불_B0M          0
이용건수_할부_B0M           0
이용건수_할부_유이자_B0M       0
이용건수_할부_무이자_B0M       0
이용건수_부분무이자_B0M        0
이용건수_CA_B0M           0
이용건수_체크_B0M           0
이용건수_카드론_B0M          0
이용금액_일시불_B0M          0
이용금액_할부_B0M           0
이용금액_할부_유이자_B0M       0
이용금액_할부_무이자_B0M       0
이용금액_부분무이자_B0M        0
이용금액_CA_B0M           0
이용금액_체크_B0M           0
이용금액_카드론_B0M          0
이용후경과월_신용             0
이용후경과월_신판             0
이용후경과월_일시불            0
이용후경과월_할부             0
이용후경과월_할부_유이자         0
이용후경과월_할부_무이자         0
이용후경과월_부분무이자          0
이용후경과월_CA             0
이용후경과월_체크             0
이용후경과월_카드론            0
이용건수_신용_R12M          0
이용건수_신판_R12M          0
이용건수_일시불_R12M         0
이용건수_할부_R12M    

In [15]:
for col in column_list:
    try:
        var_value = all_df[col].var()
        print(f"{col}: {var_value:.2f}")
    except TypeError:
        print(f"❌ 분산 계산 불가 (문자열 포함): {col}")


기준년월: 2.92
❌ 분산 계산 불가 (문자열 포함): ID
최종이용일자_기본: 15164506136473.34
최종이용일자_신판: 15290383035759.30
최종이용일자_CA: 88162643539604.62
최종이용일자_카드론: 57709158372772.21
최종이용일자_체크: 94398686619799.09
최종이용일자_일시불: 16014268555037.79
최종이용일자_할부: 81531881958688.75
이용건수_신용_B0M: 397.45
이용건수_신판_B0M: 396.95
이용건수_일시불_B0M: 393.04
이용건수_할부_B0M: 0.20
이용건수_할부_유이자_B0M: 0.03
이용건수_할부_무이자_B0M: 0.14
이용건수_부분무이자_B0M: 0.00
이용건수_CA_B0M: 0.16
이용건수_체크_B0M: 80.43
이용건수_카드론_B0M: 0.00
이용금액_일시불_B0M: 21094276.07
이용금액_할부_B0M: 2220701.16
이용금액_할부_유이자_B0M: 514047.01
이용금액_할부_무이자_B0M: 1354412.20
이용금액_부분무이자_B0M: 0.00
이용금액_CA_B0M: 4662026.63
이용금액_체크_B0M: 3700739.70
이용금액_카드론_B0M: 205861.13
이용후경과월_신용: 17.05
이용후경과월_신판: 17.32
이용후경과월_일시불: 16.10
이용후경과월_할부: 21.31
이용후경과월_할부_유이자: 11.53
이용후경과월_할부_무이자: 20.23
이용후경과월_부분무이자: 1.27
이용후경과월_CA: 8.36
이용후경과월_체크: 17.51
이용후경과월_카드론: 1.80
이용건수_신용_R12M: 51273.49
이용건수_신판_R12M: 51186.08
이용건수_일시불_R12M: 50377.42
이용건수_할부_R12M: 41.24
이용건수_할부_유이자_R12M: 3.98
이용건수_할부_무이자_R12M: 28.52
이용건수_부분무이자_R12M: 0.00
이용건수_CA_R12M: 12.26
이용건

- 이용금액_부분무이자_B0M, 이용건수_부분무이자_B0M 컬럼은 제거해야함

- 0 하나만 가진 데이터값은 상관관계 분석이나 추후 학습에 방해되므로 drop

In [16]:
all_df.drop(['이용건수_부분무이자_B0M', '이용금액_부분무이자_B0M'],
            axis=1, inplace=True)


### 변수간 관계파악
Anova -> ETA제곱
카이제곱 -> 크레이머스V

In [17]:
print(column_list)

['기준년월', 'ID', '최종이용일자_기본', '최종이용일자_신판', '최종이용일자_CA', '최종이용일자_카드론', '최종이용일자_체크', '최종이용일자_일시불', '최종이용일자_할부', '이용건수_신용_B0M', '이용건수_신판_B0M', '이용건수_일시불_B0M', '이용건수_할부_B0M', '이용건수_할부_유이자_B0M', '이용건수_할부_무이자_B0M', '이용건수_부분무이자_B0M', '이용건수_CA_B0M', '이용건수_체크_B0M', '이용건수_카드론_B0M', '이용금액_일시불_B0M', '이용금액_할부_B0M', '이용금액_할부_유이자_B0M', '이용금액_할부_무이자_B0M', '이용금액_부분무이자_B0M', '이용금액_CA_B0M', '이용금액_체크_B0M', '이용금액_카드론_B0M', '이용후경과월_신용', '이용후경과월_신판', '이용후경과월_일시불', '이용후경과월_할부', '이용후경과월_할부_유이자', '이용후경과월_할부_무이자', '이용후경과월_부분무이자', '이용후경과월_CA', '이용후경과월_체크', '이용후경과월_카드론', '이용건수_신용_R12M', '이용건수_신판_R12M', '이용건수_일시불_R12M', '이용건수_할부_R12M', '이용건수_할부_유이자_R12M', '이용건수_할부_무이자_R12M', '이용건수_부분무이자_R12M', '이용건수_CA_R12M', '이용건수_체크_R12M', '이용건수_카드론_R12M', '이용금액_일시불_R12M', '이용금액_할부_R12M', '이용금액_할부_유이자_R12M', '이용금액_할부_무이자_R12M', '이용금액_부분무이자_R12M', '이용금액_CA_R12M', '이용금액_체크_R12M', '이용금액_카드론_R12M', '최대이용금액_일시불_R12M', '최대이용금액_할부_R12M', '최대이용금액_할부_유이자_R12M', '최대이용금액_할부_무이자_R12M', '최대이용금액_부분무이자_R12M', '최대이용금액_CA_R12M', '최대이용금액_체크_R12M', '

In [18]:
# 크레이머스V와 eta를 구하기 위한 함수
def cramers_v(confusion_matrix):
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    return np.sqrt(chi2 / (n * (min(k, r) - 1)))

def eta_squared(anova_ss_between, total_ss):
    return anova_ss_between / total_ss if total_ss != 0 else np.nan

In [19]:
# 수치형/범주형 자동 구분
num_cols = all_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = all_df.select_dtypes(exclude=[np.number]).columns.tolist()

In [20]:
print(cat_cols)

['Segment', 'ID']


In [21]:
print(num_cols)

['기준년월', '최종이용일자_기본', '최종이용일자_신판', '최종이용일자_CA', '최종이용일자_카드론', '최종이용일자_체크', '최종이용일자_일시불', '최종이용일자_할부', '이용건수_신용_B0M', '이용건수_신판_B0M', '이용건수_일시불_B0M', '이용건수_할부_B0M', '이용건수_할부_유이자_B0M', '이용건수_할부_무이자_B0M', '이용건수_CA_B0M', '이용건수_체크_B0M', '이용건수_카드론_B0M', '이용금액_일시불_B0M', '이용금액_할부_B0M', '이용금액_할부_유이자_B0M', '이용금액_할부_무이자_B0M', '이용금액_CA_B0M', '이용금액_체크_B0M', '이용금액_카드론_B0M', '이용후경과월_신용', '이용후경과월_신판', '이용후경과월_일시불', '이용후경과월_할부', '이용후경과월_할부_유이자', '이용후경과월_할부_무이자', '이용후경과월_부분무이자', '이용후경과월_CA', '이용후경과월_체크', '이용후경과월_카드론', '이용건수_신용_R12M', '이용건수_신판_R12M', '이용건수_일시불_R12M', '이용건수_할부_R12M', '이용건수_할부_유이자_R12M', '이용건수_할부_무이자_R12M', '이용건수_부분무이자_R12M', '이용건수_CA_R12M', '이용건수_체크_R12M', '이용건수_카드론_R12M', '이용금액_일시불_R12M', '이용금액_할부_R12M', '이용금액_할부_유이자_R12M', '이용금액_할부_무이자_R12M', '이용금액_부분무이자_R12M', '이용금액_CA_R12M', '이용금액_체크_R12M', '이용금액_카드론_R12M', '최대이용금액_일시불_R12M', '최대이용금액_할부_R12M', '최대이용금액_할부_유이자_R12M', '최대이용금액_할부_무이자_R12M', '최대이용금액_부분무이자_R12M', '최대이용금액_CA_R12M', '최대이용금액_체크_R12M', '최대이용금액_카드론_R12M', '이용개월수_신용_R12M', '이용개월수_

### 기존의 코드

In [22]:
anova = []
chi = []

for col in all_df.columns:
    if col == 'Segment':
        continue

    if col in num_cols:
        data = all_df[[col, 'Segment']].dropna()
        groups = [data[data['Segment'] == val][col] for val in data['Segment'].unique()]
        try:
            stat = f_oneway(*groups).statistic
            ss_between = sum([(g.mean() - data[col].mean())**2 * len(g) for g in groups])
            ss_total = sum((data[col] - data[col].mean())**2)
            eta2 = eta_squared(ss_between, ss_total)
            anova.append({'변수': col, '유형': '수치형', '계수종류': 'Eta²', '상관계수': eta2})
        except:
            continue

    elif col in cat_cols:
        contingency = pd.crosstab(all_df[col], all_df['Segment'])
        if contingency.shape[0] > 1 and contingency.shape[1] > 1:
            try:
                v = cramers_v(contingency)
                chi.append({'변수': col, '유형': '범주형', '계수종류': "Cramér's V", '상관계수': v})
            except:
                continue

# 결과 정리
result_df1 = pd.DataFrame(anova)
result_df2 = pd.DataFrame(chi)
result_df1 = result_df1.sort_values(by='상관계수', ascending=False).reset_index(drop=True)
result_df2 = result_df2.sort_values(by='상관계수', ascending=False).reset_index(drop=True)

# 결과 출력
display(result_df1)
display(result_df2)

,변수,유형,계수종류,상관계수
0,이용금액_일시불_R12M,수치형,Eta²,0.355606
1,이용금액_일시불_B0M,수치형,Eta²,0.331317
2,이용금액_일시불_R6M,수치형,Eta²,0.324957
3,이용금액_일시불_R3M,수치형,Eta²,0.322906
4,이용건수_신용_R12M,수치형,Eta²,0.240197
...,...,...,...,...
122,이용건수_부분무이자_R12M,수치형,Eta²,0.000070
123,이용건수_부분무이자_R3M,수치형,Eta²,0.000068
124,이용금액_카드론_B0M,수치형,Eta²,0.000003
125,이용건수_카드론_B0M,수치형,Eta²,0.000002


,변수,유형,계수종류,상관계수
0,ID,범주형,Cramér's V,1.0


In [ ]:
pd.set_option('display.max_rows', 100)
display(result_df1)

,변수,유형,계수종류,상관계수
0,_3순위업종_이용금액,수치형,Eta²,2.549372e-01
1,_2순위업종_이용금액,수치형,Eta²,2.429217e-01
2,_2순위쇼핑업종_이용금액,수치형,Eta²,2.387084e-01
3,_1순위업종_이용금액,수치형,Eta²,2.198912e-01
4,_3순위쇼핑업종_이용금액,수치형,Eta²,2.172726e-01
5,이용가맹점수,수치형,Eta²,2.157008e-01
6,쇼핑_도소매_이용금액,수치형,Eta²,2.048195e-01
7,_1순위교통업종_이용금액,수치형,Eta²,1.715343e-01
8,쇼핑_마트_이용금액,수치형,Eta²,1.575289e-01
9,쇼핑_슈퍼마켓_이용금액,수치형,Eta²,1.560192e-01


In [ ]:
result_df1['상관계수'] = result_df1['상관계수'].round(4)

In [ ]:
display(result_df1)

,변수,유형,계수종류,상관계수
0,_3순위업종_이용금액,수치형,Eta²,0.2549
1,_2순위업종_이용금액,수치형,Eta²,0.2429
2,_2순위쇼핑업종_이용금액,수치형,Eta²,0.2387
3,_1순위업종_이용금액,수치형,Eta²,0.2199
4,_3순위쇼핑업종_이용금액,수치형,Eta²,0.2173
5,이용가맹점수,수치형,Eta²,0.2157
6,쇼핑_도소매_이용금액,수치형,Eta²,0.2048
7,_1순위교통업종_이용금액,수치형,Eta²,0.1715
8,쇼핑_마트_이용금액,수치형,Eta²,0.1575
9,쇼핑_슈퍼마켓_이용금액,수치형,Eta²,0.1560


In [23]:
cat_cols

['Segment', 'ID']

### A,B 와 CDE나누기

In [24]:
all_df.columns

Index(['Segment', '기준년월', 'ID', '최종이용일자_기본', '최종이용일자_신판', '최종이용일자_CA',
       '최종이용일자_카드론', '최종이용일자_체크', '최종이용일자_일시불', '최종이용일자_할부',
       ...
       '이용개월수_신판_R3M', '이용개월수_일시불_R3M', '이용개월수_할부_R3M', '이용개월수_할부_유이자_R3M',
       '이용개월수_할부_무이자_R3M', '이용개월수_부분무이자_R3M', '이용개월수_CA_R3M', '이용개월수_체크_R3M',
       '이용개월수_카드론_R3M', '이용가맹점수'],
      dtype='object', length=129)

In [25]:
# A, B 클래스만 필터링
ab_df = all_df[all_df['Segment'].isin(['A', 'B'])]

# C, D, E 클래스만 필터링
cde_df = all_df[all_df['Segment'].isin(['C', 'D', 'E'])]

In [26]:
ab_df

,Segment,기준년월,ID,최종이용일자_기본,최종이용일자_신판,최종이용일자_CA,최종이용일자_카드론,최종이용일자_체크,최종이용일자_일시불,최종이용일자_할부,...,이용개월수_신판_R3M,이용개월수_일시불_R3M,이용개월수_할부_R3M,이용개월수_할부_유이자_R3M,이용개월수_할부_무이자_R3M,이용개월수_부분무이자_R3M,이용개월수_CA_R3M,이용개월수_체크_R3M,이용개월수_카드론_R3M,이용가맹점수
2898,A,201807,TRAIN_002898,20180731,20180731,20180716,20171203,10101,20180731,20180723,...,3,3,3,1,3,0,2,0,0,43
5253,A,201807,TRAIN_005253,20180719,20180719,20180714,20160830,10101,20180711,20180719,...,3,3,3,0,3,0,2,0,0,28
8128,A,201807,TRAIN_008128,20180729,20180729,10101,10101,20171214,20180729,20180719,...,3,3,3,0,3,0,0,0,0,80
10808,A,201807,TRAIN_010808,20180707,20180707,20110306,10101,10101,20180707,20180610,...,3,3,1,0,1,0,0,0,0,82
14951,A,201807,TRAIN_014951,20180731,20180731,20150704,10101,10101,20180731,20180716,...,3,3,2,0,2,0,0,0,0,67
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2376373,A,201812,TRAIN_376373,20181231,20181231,10101,10101,20150109,20181231,20181224,...,3,3,3,0,3,0,0,0,0,86
2378479,A,201812,TRAIN_378479,20181231,20181231,10101,10101,10101,20181231,20180430,...,3,3,0,0,0,0,0,0,0,102
2390620,B,201812,TRAIN_390620,20181227,20181227,20181222,20171113,10101,20181227,20181216,...,3,3,3,0,3,0,3,0,0,6
2393027,A,201812,TRAIN_393027,20181208,20181208,20130608,10101,20181231,20181208,20180421,...,3,3,0,0,0,0,0,3,0,69


In [27]:
cde_df

,Segment,기준년월,ID,최종이용일자_기본,최종이용일자_신판,최종이용일자_CA,최종이용일자_카드론,최종이용일자_체크,최종이용일자_일시불,최종이용일자_할부,...,이용개월수_신판_R3M,이용개월수_일시불_R3M,이용개월수_할부_R3M,이용개월수_할부_유이자_R3M,이용개월수_할부_무이자_R3M,이용개월수_부분무이자_R3M,이용개월수_CA_R3M,이용개월수_체크_R3M,이용개월수_카드론_R3M,이용가맹점수
0,D,201807,TRAIN_000000,20180719,20180713,20180719,10101,20180203,20180709,20180713,...,3,3,1,0,1,0,3,0,0,6
1,E,201807,TRAIN_000001,20180719,20180719,20170728,20170327,10101,20180719,20171231,...,3,3,0,0,0,0,0,0,0,32
2,C,201807,TRAIN_000002,20180706,20180706,20180706,20151119,20141230,20180706,20180627,...,3,3,1,0,1,0,2,0,0,27
3,D,201807,TRAIN_000003,20180721,20180715,20180721,10101,20141111,20180704,20180715,...,3,3,2,1,1,0,3,0,0,10
4,E,201807,TRAIN_000004,20180124,20180124,10101,10101,20180512,20180124,10101,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399995,E,201812,TRAIN_399995,20181220,20181220,10101,10101,20181212,20181220,20160501,...,0,0,0,0,0,0,0,3,0,3
2399996,D,201812,TRAIN_399996,20181202,20181202,10101,20170112,10101,20181202,20180112,...,3,3,0,0,0,0,0,0,0,15
2399997,C,201812,TRAIN_399997,20181230,20181230,10101,10101,20131124,20181230,20180919,...,3,3,1,0,1,0,0,0,0,27
2399998,E,201812,TRAIN_399998,20161224,20161224,10101,10101,10101,20161224,20150122,...,0,0,0,0,0,0,0,0,0,0


In [28]:
anova = []
chi = []

for col in ab_df.columns:
    if col == 'Segment':
        continue

    if col in num_cols:
        data = ab_df[[col, 'Segment']].dropna()
        groups = [data[data['Segment'] == val][col] for val in data['Segment'].unique()]
        try:
            stat = f_oneway(*groups).statistic
            ss_between = sum([(g.mean() - data[col].mean())**2 * len(g) for g in groups])
            ss_total = sum((data[col] - data[col].mean())**2)
            eta2 = eta_squared(ss_between, ss_total)
            anova.append({'변수': col, '유형': '수치형', '계수종류': 'Eta²', '상관계수': eta2})
        except:
            continue

    elif col in cat_cols:
        contingency = pd.crosstab(ab_df[col], ab_df['Segment'])
        if contingency.shape[0] > 1 and contingency.shape[1] > 1:
            try:
                v = cramers_v(contingency)
                chi.append({'변수': col, '유형': '범주형', '계수종류': "Cramér's V", '상관계수': v})
            except:
                continue

# 결과 정리
result_df11 = pd.DataFrame(anova)
result_df12 = pd.DataFrame(chi)
result_df11 = result_df11.sort_values(by='상관계수', ascending=False).reset_index(drop=True)
result_df12 = result_df12.sort_values(by='상관계수', ascending=False).reset_index(drop=True)

# 결과 출력
display(result_df11)
display(result_df12)

/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:586: ConstantInputWarning: Each of the input arrays is constant; the F statistic is not defined or infinite
  res = hypotest_fun_out(*samples, **kwds)
/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:586: ConstantInputWarning: Each of the input arrays is constant; the F statistic is not defined or infinite
  res = hypotest_fun_out(*samples, **kwds)
/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:586: ConstantInputWarning: Each of the input arrays is constant; the F statistic is not defined or infinite
  res = hypotest_fun_out(*samples, **kwds)


,변수,유형,계수종류,상관계수
0,이용개월수_할부_무이자_R6M,수치형,Eta²,8.608794e-02
1,이용개월수_할부_R6M,수치형,Eta²,8.438067e-02
2,이용개월수_할부_무이자_R3M,수치형,Eta²,8.010630e-02
3,이용개월수_할부_무이자_R12M,수치형,Eta²,7.871694e-02
4,이용건수_할부_무이자_R3M,수치형,Eta²,7.803974e-02
...,...,...,...,...
122,최대이용금액_부분무이자_R12M,수치형,Eta²,1.201786e-07
123,기준년월,수치형,Eta²,0.000000e+00
124,이용건수_카드론_B0M,수치형,Eta²,NaN
125,이용금액_카드론_B0M,수치형,Eta²,NaN


,변수,유형,계수종류,상관계수
0,ID,범주형,Cramér's V,1.0


In [30]:
anova = []
chi = []

for col in cde_df.columns:
    if col == 'Segment':
        continue

    if col in num_cols:
        data = cde_df[[col, 'Segment']].dropna()
        groups = [data[data['Segment'] == val][col] for val in data['Segment'].unique()]
        try:
            stat = f_oneway(*groups).statistic
            ss_between = sum([(g.mean() - data[col].mean())**2 * len(g) for g in groups])
            ss_total = sum((data[col] - data[col].mean())**2)
            eta2 = eta_squared(ss_between, ss_total)
            anova.append({'변수': col, '유형': '수치형', '계수종류': 'Eta²', '상관계수': eta2})
        except:
            continue

    elif col in cat_cols:
        contingency = pd.crosstab(cde_df[col], cde_df['Segment'])
        if contingency.shape[0] > 1 and contingency.shape[1] > 1:
            try:
                v = cramers_v(contingency)
                chi.append({'변수': col, '유형': '범주형', '계수종류': "Cramér's V", '상관계수': v})
            except:
                continue

# 결과 정리
result_df21 = pd.DataFrame(anova)
result_df22 = pd.DataFrame(chi)
result_df21 = result_df21.sort_values(by='상관계수', ascending=False).reset_index(drop=True)
result_df22 = result_df22.sort_values(by='상관계수', ascending=False).reset_index(drop=True)

# 결과 출력
display(result_df21)
display(result_df22)

,변수,유형,계수종류,상관계수
0,이용금액_일시불_R12M,수치형,Eta²,0.349914
1,이용금액_일시불_B0M,수치형,Eta²,0.327886
2,이용금액_일시불_R6M,수치형,Eta²,0.321964
3,이용금액_일시불_R3M,수치형,Eta²,0.319745
4,이용건수_신용_R12M,수치형,Eta²,0.238931
...,...,...,...,...
122,이용건수_부분무이자_R12M,수치형,Eta²,0.000070
123,이용건수_부분무이자_R3M,수치형,Eta²,0.000062
124,이용금액_카드론_B0M,수치형,Eta²,0.000003
125,이용건수_카드론_B0M,수치형,Eta²,0.000001


,변수,유형,계수종류,상관계수
0,ID,범주형,Cramér's V,1.0


In [31]:
pd.set_option('display.max_rows', None)

In [32]:
result_df1['상관계수'] = result_df1['상관계수'].round(4)
result_df11['상관계수'] = result_df11['상관계수'].round(4)
result_df21['상관계수'] = result_df21['상관계수'].round(4)

In [33]:
result_df1

,변수,유형,계수종류,상관계수
0,이용금액_일시불_R12M,수치형,Eta²,0.3556
1,이용금액_일시불_B0M,수치형,Eta²,0.3313
2,이용금액_일시불_R6M,수치형,Eta²,0.3250
3,이용금액_일시불_R3M,수치형,Eta²,0.3229
4,이용건수_신용_R12M,수치형,Eta²,0.2402
5,최대이용금액_일시불_R12M,수치형,Eta²,0.2370
6,이용건수_신판_R12M,수치형,Eta²,0.2365
7,이용건수_일시불_R12M,수치형,Eta²,0.2321
8,이용가맹점수,수치형,Eta²,0.2157
9,이용건수_신용_R6M,수치형,Eta²,0.1969


In [34]:
result_df21

,변수,유형,계수종류,상관계수
0,이용금액_일시불_R12M,수치형,Eta²,0.3499
1,이용금액_일시불_B0M,수치형,Eta²,0.3279
2,이용금액_일시불_R6M,수치형,Eta²,0.3220
3,이용금액_일시불_R3M,수치형,Eta²,0.3197
4,이용건수_신용_R12M,수치형,Eta²,0.2389
5,이용건수_신판_R12M,수치형,Eta²,0.2353
6,최대이용금액_일시불_R12M,수치형,Eta²,0.2328
7,이용건수_일시불_R12M,수치형,Eta²,0.2309
8,이용가맹점수,수치형,Eta²,0.2143
9,이용건수_신용_R6M,수치형,Eta²,0.1957


In [35]:

ab_df.to_parquet('승인매출ab_df3-1.parquet', index=True)
cde_df.to_parquet('승인매출cde_df3-1.parquet', index=True)

result_df21.to_parquet('승인매출cde_df3-1_설명력.parquet', index=True)

In [36]:
ab_df.to_parquet('승인매출ab_df3-1.parquet', index=False)
cde_df.to_parquet('승인매출cde_df3-1.parquet', index=False)

In [37]:
from scipy.stats import ttest_ind
def cohen_d(x, y):
    """두 그룹 간 효과 크기(Cohen's d) 계산"""
    nx = len(x)
    ny = len(y)
    dof = nx + ny - 2
    pooled_std = np.sqrt(((nx - 1)*x.std()**2 + (ny - 1)*y.std()**2) / dof)
    return (x.mean() - y.mean()) / pooled_std

def ttest_ab_multiple_verbose_abs_sorted(df, feature_cols):
    """
    A vs B 그룹 간 연속형 변수들에 대해 t-test, p-value, Cohen's d, 유의성 표시 포함한 결과 반환
    Cohen's d 절댓값 기준 내림차순 정렬
    """
    results = []

    for col in feature_cols:
        group_a = df[df['Segment'] == 'A'][col].dropna()
        group_b = df[df['Segment'] == 'B'][col].dropna()

        t_stat, p_val = ttest_ind(group_a, group_b, equal_var=False)
        d = cohen_d(group_a, group_b)

        # 유의성 표시
        if p_val < 0.001:
            sig = '***'
        elif p_val < 0.01:
            sig = '**'
        elif p_val < 0.05:
            sig = '*'
        else:
            sig = 'ns'

        results.append({
            'feature': col,
            't-statistic': round(t_stat, 4),
            'p-value': round(p_val, 4),
            'Cohen\'s d': round(d, 4),
            'abs(Cohen\'s d)': abs(round(d, 4)),
            'significance': sig
        })

    df_result = pd.DataFrame(results)
    df_result = df_result.sort_values('abs(Cohen\'s d)', ascending=False).reset_index(drop=True)
    df_result = df_result.drop(columns=['abs(Cohen\'s d)'])  # 필요 없으면 제거

    return df_result

In [38]:
ttest_results = ttest_ab_multiple_verbose_abs_sorted(ab_df, num_cols)
ttest_results

/tmp/ipython-input-37-1371196798.py:8: RuntimeWarning: invalid value encountered in scalar divide
  return (x.mean() - y.mean()) / pooled_std
/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


,feature,t-statistic,p-value,Cohen's d,significance
0,이용개월수_할부_무이자_R6M,-13.5899,0.0000,-0.9147,***
1,이용개월수_할부_R6M,-14.4447,0.0000,-0.9047,***
2,이용개월수_할부_무이자_R3M,-12.8663,0.0000,-0.8795,***
3,이용개월수_할부_무이자_R12M,-17.5522,0.0000,-0.8712,***
4,이용건수_할부_무이자_R3M,-11.9087,0.0000,-0.8671,***
5,이용건수_할부_무이자_R6M,-11.0264,0.0000,-0.8513,***
6,이용개월수_할부_R3M,-13.0551,0.0000,-0.8483,***
7,이용개월수_할부_R12M,-15.8418,0.0000,-0.8462,***
8,이용건수_할부_무이자_B0M,-11.0421,0.0000,-0.8438,***
9,이용건수_할부_R3M,-11.5094,0.0000,-0.8371,***


In [39]:

pd.set_option('display.float_format', '{:.5f}'.format)

# Cramér's V
def cramers_v(confusion_matrix):
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    return np.sqrt(chi2 / (n * (min(k, r) - 1)))

# Eta² / t-test
def eta_squared_from_t(t, n1, n2):
    df = n1 + n2 - 2
    return t**2 / (t**2 + df) if df > 0 else np.nan

In [40]:
def effect_sizes(df):
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

    num_cols = [col for col in num_cols if col != 'Segment']
    cat_cols = [col for col in cat_cols if col != 'Segment']

    ttest_results = []
    chi_results = []

    for col in df.columns:
        if col == 'Segment':
            continue

        # 수치형 변수
        if col in num_cols:
            data = df[[col, 'Segment']].dropna()
            group1 = data[data['Segment'] == data['Segment'].unique()[0]][col]
            group2 = data[data['Segment'] == data['Segment'].unique()[1]][col]
            try:
                t_stat, _ = ttest_ind(group1, group2, equal_var=False)
                eta2 = eta_squared_from_t(t_stat, len(group1), len(group2))
                ttest_results.append({
                    '변수': col, '유형': '수치형',
                    '계수종류': 'Eta²',
                    '상관계수': eta2
                })
            except:
                continue

        # 범주형 변수
        elif col in cat_cols:
            contingency = pd.crosstab(df[col], df['Segment'])
            if contingency.shape[0] > 1 and contingency.shape[1] > 1:
                try:
                    v = cramers_v(contingency)
                    chi_results.append({
                        '변수': col, '유형': '범주형',
                        '계수종류': "Cramér's V",
                        '상관계수': v
                    })
                except:
                    continue

    result_df1 = pd.DataFrame(ttest_results).sort_values(by='상관계수', ascending=False).reset_index(drop=True)
    result_df2 = pd.DataFrame(chi_results).sort_values(by='상관계수', ascending=False).reset_index(drop=True)

    return result_df1, result_df2


In [41]:
result1, result2 = effect_sizes(ab_df)
display(result1)
display(result2)

/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment 

,변수,유형,계수종류,상관계수
0,이용후경과월_할부_무이자,수치형,Eta²,0.23384
1,이용후경과월_할부,수치형,Eta²,0.22677
2,이용개월수_할부_무이자_R12M,수치형,Eta²,0.21664
3,이용개월수_할부_R12M,수치형,Eta²,0.18386
4,이용개월수_할부_R6M,수치형,Eta²,0.15775
5,이용개월수_할부_무이자_R6M,수치형,Eta²,0.14221
6,이용개월수_할부_R3M,수치형,Eta²,0.13269
7,이용개월수_할부_무이자_R3M,수치형,Eta²,0.12938
8,이용건수_할부_무이자_R3M,수치형,Eta²,0.11293
9,이용건수_할부_R3M,수치형,Eta²,0.10627


,변수,유형,계수종류,상관계수
0,ID,범주형,Cramér's V,1.00000


In [43]:
#result1과 result2, 그리고 ttest_results를 각각 parquet으로 저장한다.
result1.to_parquet('/content/drive/MyDrive/승인매출ab_df3-1_설명력.parquet', index=False)
ttest_results.to_parquet('/content/drive/MyDrive/ttest_results3-1.parquet', index=False)